# 02. Grad-CAM++ 시각화
DenseNet-121이 흉부 X-Ray에서 **어느 부분을 보고** 질환을 판단했는지 히트맵으로 시각화

### Grad-CAM++ 원리
1. 이미지를 모델에 통과시킴 (Forward pass)
2. 특정 질환에 대한 출력값을 역전파 (Backward pass)
3. 마지막 합성곱 층(norm5)의 gradient 추출
4. pixel-wise 가중치 계산 -> 히트맵 생성
5. 히트맵을 원본 이미지에 오버레이

### 파이프라인 위치
```
DenseNet-121 (질환 분류) -> [Grad-CAM++ 히트맵] -> PubMedBERT RAG -> Bedrock 소견서
```

In [ ]:
!pip install grad-cam -q

import os, io, json, tarfile, numpy as np, pandas as pd
import torch, torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import boto3
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image


class MultiLabelTarget:
    """멀티레이블 분류에서 특정 질환 인덱스의 출력을 스칼라로 반환"""
    def __init__(self, category):
        self.category = category
    def __call__(self, model_output):
        if len(model_output.shape) == 1:
            return model_output[self.category]
        return model_output[:, self.category].sum()


print('Import 완료!')

## 1. 설정

In [ ]:
S3_BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
TRAINING_JOB_NAME = 'densenet121-mimic-cxr-v1'

LABEL_COLS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity',
    'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia',
    'Pneumothorax', 'Support Devices'
]
NUM_CLASSES = 14
NUM_VISUALIZE = 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
s3 = boto3.client('s3')
print(f'Device: {device}')

## 2. 모델 다운로드 & 로드

In [ ]:
print('모델 다운로드 중...')
s3.download_file(S3_BUCKET, f'output/{TRAINING_JOB_NAME}/output/model.tar.gz', '/tmp/model.tar.gz')
with tarfile.open('/tmp/model.tar.gz', 'r:gz') as tar:
    tar.extractall('/tmp/model')

with open('/tmp/model/results.json') as f:
    results = json.load(f)
print(f'Mean AUROC: {results["mean_auroc"]:.4f}')
print(f'학습 시간: {results["training_time_minutes"]:.1f}분')

model = models.densenet121(weights=None)
model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)
model.load_state_dict(torch.load('/tmp/model/best_model.pth', map_location=device))
model = model.to(device)
model.eval()
print('모델 로드 완료!')

## 3. Grad-CAM++ 설정
타겟 레이어: `model.features.norm5` (denseblock4 뒤 BatchNorm, 의료영상 표준)

In [ ]:
cam = GradCAMPlusPlus(model=model, target_layers=[model.features.norm5])
print('Grad-CAM++ 초기화 완료')

## 4. 테스트 이미지 선택 (양성 라벨 많은 순)

In [ ]:
s3.download_file(S3_BUCKET, 'preprocessing/p10_train_ready_resplit.csv', '/tmp/test.csv')
df = pd.read_csv('/tmp/test.csv')
test_df = df[df['split'] == 'test'].copy()
test_df['positive_count'] = test_df[LABEL_COLS].sum(axis=1)
test_df = test_df.sort_values('positive_count', ascending=False).head(NUM_VISUALIZE)
print(f'시각화 대상: {len(test_df)}개 이미지')

## 5. 전처리 함수

In [ ]:
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def load_image_from_s3(image_path):
    s3_key = f'data/p10_pa/{image_path}'
    response = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    image = Image.open(io.BytesIO(response['Body'].read())).convert('RGB')
    original_np = np.array(image.resize((224, 224))).astype(np.float32) / 255.0
    input_tensor = inference_transform(image).unsqueeze(0)
    return original_np, input_tensor

print('전처리 함수 준비 완료')

## 6. Grad-CAM++ 히트맵 생성
- 초록(●): 실제 양성 | 빨강(○): 모델 예측만 양성

In [ ]:
output_dir = '/tmp/gradcam_results'
os.makedirs(output_dir, exist_ok=True)
all_results = []

for i, (_, row) in enumerate(test_df.iterrows()):
    print(f'\n[{i+1}/{len(test_df)}] {row["image_path"].split("/")[-1]}')

    original_np, input_tensor = load_image_from_s3(row['image_path'])
    input_tensor = input_tensor.to(device)

    with torch.no_grad():
        probs = torch.sigmoid(model(input_tensor)).cpu().numpy()[0]

    true_pos = [LABEL_COLS[j] for j in range(NUM_CLASSES) if row[LABEL_COLS[j]] == 1]
    pred_top = sorted(range(NUM_CLASSES), key=lambda j: -probs[j])

    viz_diseases = list(dict.fromkeys(true_pos + [LABEL_COLS[j] for j in pred_top[:3]]))[:6]
    if not viz_diseases:
        viz_diseases = [LABEL_COLS[j] for j in pred_top[:3]]

    print(f'  실제: {true_pos if true_pos else ["No Finding"]}')
    print(f'  예측 Top3: {[(LABEL_COLS[j], f"{probs[j]:.2f}") for j in pred_top[:3]]}')

    n_cols = len(viz_diseases) + 1
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

    axes[0].imshow(original_np)
    axes[0].set_title('Original', fontsize=11, fontweight='bold')
    true_text = '\n'.join([f'+ {d}' for d in true_pos]) if true_pos else 'No Finding'
    axes[0].text(5, 210, true_text, fontsize=7, color='lime',
                 bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
    axes[0].axis('off')

    for idx, disease in enumerate(viz_diseases):
        d_idx = LABEL_COLS.index(disease)
        targets = [MultiLabelTarget(d_idx)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]
        visualization = show_cam_on_image(original_np, grayscale_cam, use_rgb=True)

        ax = axes[idx + 1]
        ax.imshow(visualization)
        is_true = disease in true_pos
        color = 'green' if is_true else 'red'
        marker = chr(9679) if is_true else chr(9675)
        ax.set_title(f'{marker} {disease}\nP={probs[d_idx]:.2f}',
                     fontsize=9, color=color, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f'[{i+1}/{len(test_df)}] {row["image_path"].split("/")[-1]}', fontsize=10)
    plt.tight_layout()
    save_path = os.path.join(output_dir, f'gradcam_{i+1:02d}.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

    all_results.append({
        'image_path': row['image_path'],
        'true_labels': true_pos,
        'predictions': {LABEL_COLS[j]: float(probs[j]) for j in range(NUM_CLASSES)}
    })

print(f'\n시각화 완료! {len(all_results)}개 이미지')


## 7. 결과 S3 업로드

In [ ]:
results_json_path = os.path.join(output_dir, 'gradcam_results.json')
with open(results_json_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

s3_prefix = f'output/{TRAINING_JOB_NAME}/gradcam'
for filename in os.listdir(output_dir):
    local_path = os.path.join(output_dir, filename)
    s3.upload_file(local_path, S3_BUCKET, f'{s3_prefix}/{filename}')
    print(f'  업로드: {filename}')

print(f'\nS3 저장 완료: s3://{S3_BUCKET}/{s3_prefix}/')

## 8. GPU 메모리 정리

In [ ]:
del model, cam
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('정리 완료!')